# DL-01 · PyTorch Foundations and Training Loops
### Section 04 — From validated ML experiments to runnable deep learning

**Prerequisites:** FND-04, CML-02, MLE-01, MLE-02, and the Classical ML checkpoint.
You should understand loss, gradients, train/validation/test roles, metrics,
leakage, and reproducibility. **Estimated time:** 4–6 hours.

This notebook teaches the framework mechanics needed by every later neural
module. It deliberately comes before the NumPy derivation: first learn to inspect
tensors and run a correct experiment; then Lesson DL-02 opens the black box.


> **Canonical learner route · module DL-01 of 62**
>
> Required prerequisites: **FND-04, CML-02, MLE-02, MLE-06, EVAL-01** · Previous: **MLE-06** ·
> Next after mastery: **DL-02** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning Objectives

- Reason about tensor dtype, shape, device, and batch dimensions.
- Build `Dataset` and `DataLoader` objects without mixing data partitions.
- Use autograd and explain `zero_grad → forward → loss → backward → step`.
- Write separate training and evaluation loops with correct model modes.
- Save a `state_dict` checkpoint plus the metadata needed to reproduce it.
- Diagnose shape, device, gradient, and leakage failures.


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review PRE-01 through PRE-04 for
worked explanations of these symbols.
</details>


## Student Lesson Companion · DL-01 · PyTorch Foundations and Training Loops

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |

### Code walkthrough and expected-result contract

For the first implementation, annotate: inputs → shapes/units → initialization →
central computation → intermediate output → final output → verification. Before
execution, record the expected value, range, or shape and what it would mean.

Distinguish these outcomes:

| Outcome | Interpretation | Next action |
|---|---|---|
| Exception, non-finite value, impossible shape | Code or data-contract failure | Inspect the first violated boundary |
| Output has valid type/shape but weak metric | Experiment ran; method may be poor | Diagnose data, assumptions, and baseline comparison |
| Strong metric on training data only | Insufficient evidence | Evaluate with the declared validation design |
| Expected output on untouched data | Supports one scoped claim | Record limitations; do not generalize beyond evidence |

### Debugging table

| Symptom | Likely cause | Inspect | Scoped fix |
|---|---|---|---|
| Shape/type error | Interface mismatch | Shapes, dtypes, feature names | Repair the boundary, not downstream symptoms |
| NaN/Inf or divergence | Invalid input or unstable update | Raw values, loss, gradients, learning rate | Clean/validate input or change one optimization control |
| Implausibly strong result | Leakage or invalid split | Fit boundaries, timestamps, duplicate entities | Rebuild the evaluation path before tuning |
| Different repeated result | Uncontrolled state | Seeds, data order, train/eval mode, versions | Record and control randomness intentionally |
| Plausible output but poor result | Wrong assumptions or representation | Baseline, slices, residuals/errors | Prefer a justified alternative; do not debug valid code as broken |


## 2 · Historical Motivation

NumPy exposes numerical operations but does not automatically differentiate an
arbitrary program or manage accelerators. Modern frameworks combine tensor
kernels, reverse-mode autodiff, reusable layers, data loading, and serialization.
PyTorch's eager execution made the training program inspectable with ordinary
Python debugging—a useful fit for learning and production research.


## 3 · Intuition and Visual Understanding

A tensor is an array plus a contract: **shape, dtype, device, and gradient role**.
A batch `X` shaped `(32, 64)` means 32 examples, each with 64 features. A linear
layer with 10 outputs maps it to `(32, 10)`; the first dimension remains the
batch. Write shapes beside every arrow before writing model code.

```text
Dataset → DataLoader → batch X,y → model → logits → loss
                                  ↑                ↓
                                  └──── optimizer ← gradients
```


## 4 · Mathematical Foundations

For batch $X\in\mathbb R^{B\times d}$, weights
$W\in\mathbb R^{d\times c}$, and bias $b\in\mathbb R^c$:
$$Z=XW+b,\qquad Z\in\mathbb R^{B\times c}.$$

**Read aloud:** batch inputs times weights plus a broadcast bias gives one logit
per class for every example. $B$ is batch size, $d$ feature count, and $c$ class
count. With `(B,d)=(4,3)` and `c=2`, `(4,3)@(3,2)+(2,)` gives `(4,2)`.
Cross-entropy expects raw logits, not probabilities; it applies a stable log-softmax.


In [ ]:
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

seed_everything()
X = torch.randn(12, 4, dtype=torch.float32)
y = torch.tensor([0, 1, 2] * 4, dtype=torch.long)
print("X", X.shape, X.dtype, X.device, "y", y.shape, y.dtype)


### 4.1 Sequence-tensor bridge for later language models

Image and tabular batches often use `(B,d)`. Language models add a sequence axis:
`(B,T,C)`, where $B$ is batch size, $T$ is tokens per training window, and $C$ is
the vector width for each token. Splitting $C$ across $H$ attention heads produces
`(B,H,T,D)`, where $D=C/H` is the width of one head.

A next-token target is not a separate label column. For token IDs
`[4, 9, 2, 7, 1]`, an input window `[4, 9, 2, 7]` has target `[9, 2, 7, 1]`.
This is **teacher forcing**: at every position, the model sees real earlier tokens
and predicts the following one. A causal mask hides positions to the right so a
prediction cannot copy its answer from the future.

Softmax converts one row of logits into probabilities. PyTorch cross-entropy uses
a numerically stable log-softmax internally, so pass raw logits rather than applying
softmax first. Broadcasting lets a `(C,)` position or bias vector combine with all
`B × T` token vectors; always write the intended expanded shape before relying on it.


In [ ]:
# Shape drill used by RNN, attention, Transformer, and language-model lessons.
B, T, C, H, V = 2, 4, 8, 2, 11
token_ids = torch.tensor([[4, 9, 2, 7, 1], [3, 5, 8, 6, 0]])
inputs, targets = token_ids[:, :-1], token_ids[:, 1:]
hidden = torch.randn(B, T, C)
qkv = torch.randn(B, T, 3 * C)
query, key, value = qkv.chunk(3, dim=-1)
query = query.view(B, T, H, C // H).transpose(1, 2)
scores = query @ query.transpose(-2, -1)
logits = torch.randn(B, T, V)
loss = nn.functional.cross_entropy(logits.reshape(B * T, V), targets.reshape(B * T))

print("inputs / targets:", inputs.shape, targets.shape, "(B,T)")
print("hidden:", hidden.shape, "(B,T,C)")
print("one split Q tensor:", query.shape, "(B,H,T,D)")
print("attention scores:", scores.shape, "(B,H,T,T)")
print("logits / scalar loss:", logits.shape, loss.shape)
assert inputs[0].tolist() == [4, 9, 2, 7]
assert targets[0].tolist() == [9, 2, 7, 1]


## 5 · Manual Implementation from Scratch

Autograd records tensor operations whose inputs require gradients. The scalar
loss starts the reverse pass; each parameter accumulates its derivative. Gradients
accumulate by design, so clear them before every update.


In [ ]:
w = torch.tensor(2.0, requires_grad=True)
loss = (w - 5.0) ** 2
loss.backward()
print("loss", loss.item(), "gradient", w.grad.item())
assert w.grad.item() == -6.0


## 6 · Visualization

Inspect the learning curve, but never use training loss alone as evidence of
generalization. Record training and validation metrics separately at every epoch.


In [ ]:
loader = DataLoader(TensorDataset(X, y), batch_size=4, shuffle=True,
                    generator=torch.Generator().manual_seed(42))
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()
history = []
for epoch in range(8):
    model.train(); total = 0.0
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward(); optimizer.step()
        total += loss.item() * len(xb)
    history.append(total / len(loader.dataset))
print("loss history", [round(v, 3) for v in history])


## 7 · Failure Modes and Common Mistakes

- Passing one-hot labels to a loss expecting integer class indices.
- Applying softmax before `CrossEntropyLoss`.
- Forgetting `zero_grad`, `model.train()`, `model.eval()`, or `torch.no_grad()`.
- Evaluating on shuffled or transformed data with inconsistent preprocessing.
- Moving the model to an accelerator but leaving a batch on the CPU.
- Squeezing away the batch dimension when batch size is one.


## 8 · Library Implementation

Evaluation is a different program: disable training-only behavior and gradient
recording, collect predictions, then calculate the metric over the whole split.


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X)
    predictions = logits.argmax(dim=1)
accuracy = (predictions == y).float().mean().item()
print("evaluation accuracy", round(accuracy, 3))


## 9 · Realistic Case Study

In a digit classifier, split by independent image before fitting normalization.
Track seed, data fingerprint, split indices, architecture, optimizer, learning
rate, epoch, selected checkpoint, and test-use count. The Deep Learning checkpoint
later applies this contract to real images.


## 10 · Production and Learning Considerations

Save weights and metadata, not an unexplained Python object. A deployable artifact
must include input shape/dtype, preprocessing, class mapping, framework version,
code/data version, and selection metric. Load it on CPU in a clean process as a test.


## 11 · Tradeoff Analysis

Larger batches improve hardware utilization but change optimization and memory
use. Accelerators reduce training time but increase environment complexity.
Framework convenience speeds development, while scratch implementations remain
valuable for understanding and custom operations.


## 12 · Readiness and Interview Preparation

You are ready when you can trace shapes, explain why logits—not softmax outputs—go
into cross-entropy, write train/eval loops from memory, and diagnose a model that
changes its validation predictions between identical runs.


## 13 · Teach-Back

Explain the lifecycle of one batch without framework jargon. Then explain why
validation must use `eval()` and `no_grad()`, and why neither replaces a correct
data split.


## 14 · Exercises, Self-Check, and Solutions

1. **Guided (15 min):** predict shapes through `Linear(4,8) → ReLU → Linear(8,3)`
   for batch 32. Self-check: `(32,4) → (32,8) → (32,8) → (32,3)`.
2. **Independent (25 min):** write an evaluation function returning loss and
   accuracy. It must use `eval()` and `no_grad()` and aggregate by example count.
3. **Challenge (45 min):** save/load a `state_dict` and prove predictions match.
   Record seed, shapes, classes, and PyTorch version beside it.

Common mistakes: averaging batch averages without weighting, softmax before
cross-entropy, and changing preprocessing between training and evaluation.
Readiness threshold: 80% plus a correct train/eval-loop teach-back.


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **DL-01 · PyTorch Foundations and Training Loops** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember DL-01 · PyTorch Foundations and Training Loops: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · DL-01

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
